# Classification with Imbalanced Data
### A practical, end-to-end guide for ML practitioners

**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — Kaggle (`mlg-ulb/creditcardfraud`)  
**Stack:** Python · scikit-learn · imbalanced-learn · XGBoost · pandas · matplotlib

---

## What this notebook covers

| # | Topic |
|---|-------|
| 1 | The accuracy paradox — why accuracy lies on imbalanced data |
| 2 | The right metrics: Recall, Precision, F-beta, PR-AUC vs ROC-AUC |
| 3 | Resampling: Random Over/Undersampling, SMOTE, ADASYN, SMOTETomek |
| 4 | Cost-sensitive learning: class weights in the loss function |
| 5 | Threshold tuning: moving beyond the 0.5 default |
| 6 | Leakage-proof pipelines with `imblearn.Pipeline` + `StratifiedKFold` |
| 7 | Final comparison table across all methods |

---

> **Core principle:**  
> Most imbalanced dataset problems don't need exotic solutions.  
> They need you to **stop using accuracy**, tune the threshold, and apply class weights.  
> SMOTE helps at the margin — but only after the fundamentals are right.


---
## 1. Setup & Imports


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, ConfusionMatrixDisplay
)
import xgboost as xgb

# imblearn — install with: pip install imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN, KMeansSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline  # resampler-aware pipeline

print("All libraries imported successfully.")


---
## 2. The Accuracy Paradox — Building the Intuition

Before touching the real dataset, we build the core intuition with a synthetic example.

We create a binary classification problem where **90% of samples belong to class 1**
and only **10% to class 0** (the minority class).

A naive model that **always predicts class 1** will score **90% accuracy** — without learning anything useful.
This is the **accuracy paradox**, and it is the central reason why accuracy is the wrong metric for imbalanced problems.


In [ ]:
# Create a synthetic imbalanced dataset
# weights=[0.1, 0.9] means 10% minority, 90% majority
X_toy, y_toy = make_classification(
    n_classes=2,
    class_sep=2,
    weights=[0.1, 0.9],
    n_informative=3,
    n_redundant=1,
    flip_y=0,
    n_features=20,
    n_clusters_per_class=1,
    n_samples=1000,
    random_state=10
)

print("Class distribution:")
unique, counts = np.unique(y_toy, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} samples ({cnt/len(y_toy)*100:.1f}%)")


In [ ]:
# Visualise the class imbalance
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(
    ['Minority (Class 0)', 'Majority (Class 1)'],
    [sum(y_toy == 0), sum(y_toy == 1)],
    color=['#e74c3c', '#3498db'], edgecolor='white', linewidth=1.5
)
ax.set_title('Toy Dataset — Class Distribution', fontweight='bold')
ax.set_ylabel('Sample Count')
for bar, val in zip(ax.patches, [sum(y_toy == 0), sum(y_toy == 1)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Train SVM on the toy dataset — default settings
X_toy_train, X_toy_test, y_toy_train, y_toy_test = train_test_split(
    X_toy, y_toy, test_size=0.2, stratify=y_toy, random_state=99
)

scaler_toy = StandardScaler()
X_toy_train_sc = scaler_toy.fit_transform(X_toy_train)
X_toy_test_sc  = scaler_toy.transform(X_toy_test)

svm_toy = SVC(probability=True)
svm_toy.fit(X_toy_train_sc, y_toy_train)
y_toy_pred = svm_toy.predict(X_toy_test_sc)

print("=== Toy Dataset — SVM, default settings ===")
print(f"Accuracy  : {accuracy_score(y_toy_test, y_toy_pred):.3f}  <- looks great, but...")
print(f"Recall    : {recall_score(y_toy_test, y_toy_pred):.3f}  <- how many minority cases did we catch?")
print(f"Precision : {precision_score(y_toy_test, y_toy_pred):.3f}")
print(f"F1        : {f1_score(y_toy_test, y_toy_pred):.3f}")
print()
print(classification_report(y_toy_test, y_toy_pred))


### Observation

The toy dataset has good class separation (`class_sep=2`), so metrics look reasonable here.
In a real-world setting with noisy, overlapping classes, the model collapses toward predicting
the majority class — achieving high accuracy while completely failing on what matters.

> **Takeaway:** Always inspect Recall and Precision on the **minority class**, never just overall accuracy.


---
## 3. The Real Dataset — Credit Card Fraud Detection

**Source:** [Kaggle — mlg-ulb/creditcardfraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

### Download instructions

```bash
# Option 1: Kaggle CLI (recommended)
pip install kaggle
kaggle datasets download -d mlg-ulb/creditcardfraud
unzip creditcardfraud.zip

# Option 2: Manual — download creditcard.csv from Kaggle and place it in this directory
```

### Dataset description

| Property | Value |
|----------|-------|
| Total transactions | 284,807 |
| Fraud cases | 492 |
| Fraud rate | **0.172%** — extreme imbalance |
| Features V1–V28 | PCA-transformed (anonymised) |
| `Amount` | Transaction value (not transformed) |
| `Time` | Seconds since first transaction (we drop this) |
| `Class` | Target: 0 = legitimate, 1 = fraud |

Kaggle's own documentation notes:
> *"Given the class imbalance ratio, we recommend measuring accuracy using the Area Under
> the Precision-Recall Curve (AUPRC). Confusion matrix accuracy is not meaningful for unbalanced classification."*


In [ ]:
# Load the dataset
# Make sure creditcard.csv is in the same directory as this notebook
df = pd.read_csv('creditcard.csv')

print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Class'].value_counts())
print(f"\nFraud rate: {df['Class'].mean()*100:.4f}%")


---
## 4. Exploratory Data Analysis

Before modelling, we understand the data: its shape, missing values,
feature distributions, and the severity of class imbalance.


In [ ]:
# Basic info
df.info()


In [ ]:
# Summary statistics
df.describe().T.round(3)


In [ ]:
# Missing values check
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values — clean dataset.")
else:
    print(missing[missing > 0])


In [ ]:
# Visualise class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts()
axes[0].bar(
    ['Legitimate (0)', 'Fraud (1)'],
    counts.values,
    color=['#3498db', '#e74c3c'], edgecolor='white', linewidth=1.5
)
axes[0].set_title('Class Distribution (absolute)', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontweight='bold')

axes[1].pie(
    counts.values,
    labels=[f'Legitimate\n({counts[0]:,})', f'Fraud\n({counts[1]})'],
    autopct='%1.3f%%',
    colors=['#3498db', '#e74c3c'],
    startangle=90,
    textprops={'fontsize': 11}
)
axes[1].set_title('Class Distribution (proportional)', fontweight='bold')

plt.suptitle('Credit Card Fraud — Extreme Class Imbalance (0.172% fraud)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Transaction Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cls, color, label in [(0, '#3498db', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    axes[0].hist(df[df['Class'] == cls]['Amount'], bins=60,
                 alpha=0.7, color=color, label=label, density=True)
axes[0].set_xlabel('Transaction Amount (EUR)')
axes[0].set_ylabel('Density')
axes[0].set_title('Amount Distribution by Class')
axes[0].legend()
axes[0].set_xlim(0, 2500)

axes[1].boxplot(
    [df[df['Class'] == 0]['Amount'], df[df['Class'] == 1]['Amount']],
    labels=['Legitimate', 'Fraud'],
    patch_artist=True,
    boxprops=dict(facecolor='#3498db', alpha=0.6),
)
axes[1].set_ylabel('Transaction Amount (EUR)')
axes[1].set_title('Amount Boxplot by Class')

plt.tight_layout()
plt.show()


In [ ]:
# Top features correlated with the fraud label
corr = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = corr.head(14).index.tolist()

signed_corr = df[top_features].corrwith(df['Class'])
colors_corr = ['#e74c3c' if c > 0.2 else '#3498db' for c in signed_corr]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(top_features[::-1], corr[top_features[::-1]], color=colors_corr[::-1], alpha=0.85)
ax.set_xlabel('|Correlation with Class|')
ax.set_title('Top 14 Features by Absolute Correlation with Fraud Label', fontweight='bold')
ax.axvline(0.2, color='gray', linestyle='--', alpha=0.5, label='threshold = 0.2')
ax.legend()
plt.tight_layout()
plt.show()

print("Top 5 most correlated features:", top_features[:5])


---
## 5. Preprocessing & Train/Test Split

### Key decisions

| Decision | Reason |
|----------|--------|
| Drop `Time` | Sequential counter — no generalisation signal for fraud |
| Scale `Amount` | V1–V28 are already PCA-scaled; Amount spans a wide raw range |
| `stratify=y` in split | Preserves fraud ratio in both train and test sets |
| **No resampling before split** | Any resampling here would be **data leakage** |


In [ ]:
# Drop Time, keep Amount and V1-V28
df_model = df.drop(columns=['Time'])

X = df_model.drop('Class', axis=1)
y = df_model['Class']

# Stratified split — preserves class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train size : {X_train.shape[0]:,} samples")
print(f"Test size  : {X_test.shape[0]:,} samples")
print(f"\nTrain fraud rate: {y_train.mean()*100:.4f}%")
print(f"Test  fraud rate: {y_test.mean()*100:.4f}%")
print("\nStratification confirmed: fraud ratio preserved in both sets.")


In [ ]:
# Scale Amount only (V1-V28 already normalised by PCA)
X_train = X_train.copy()
X_test  = X_test.copy()

scaler = StandardScaler()
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount']  = scaler.transform(X_test[['Amount']])  # fit on train only

print("Scaling complete.")
print(f"Amount train — mean: {X_train['Amount'].mean():.4f}, std: {X_train['Amount'].std():.4f}")


---
## 6. Evaluation Helper

We define a unified evaluation function that computes all relevant metrics.

### Metric reference

| Metric | Formula | When it matters |
|--------|---------|-----------------|
| **Accuracy** | (TP+TN) / all | ❌ Misleading on imbalanced data |
| **Recall** | TP / (TP+FN) | ✅ When missing a fraud is costly |
| **Precision** | TP / (TP+FP) | ✅ When false alarms are costly |
| **F1** | harmonic mean(P, R) | ✅ Balanced trade-off |
| **F2 (β=2)** | weighted toward recall | ✅ Fraud/fault detection — missing is worse than a false alarm |
| **ROC-AUC** | TPR vs FPR area | ⚠️ Optimistic — FPR is inflated by the huge TN count |
| **PR-AUC** | Precision vs Recall area | ✅ Best single metric for imbalanced problems |

### PR-AUC vs ROC-AUC — the key distinction

ROC-AUC uses **False Positive Rate = FP / (FP + TN)**.  
With 284,000 legitimate transactions, TN is enormous — FPR stays tiny even with many false positives.
This makes ROC-AUC look flattering.

PR-AUC uses only **TP, FP, FN** — it is not diluted by the majority class.  
A random classifier on 0.17% fraud has **PR-AUC ≈ 0.0017**, not 0.5.


In [ ]:
def evaluate_model(y_true, y_pred, y_prob=None, model_name="Model", verbose=True):
    """
    Compute all relevant metrics for an imbalanced classification problem.

    Parameters
    ----------
    y_true     : ground truth labels
    y_pred     : binary predictions (at chosen threshold)
    y_prob     : predicted probabilities for positive class (enables AUC metrics)
    model_name : label for display
    verbose    : print report and confusion matrix if True

    Returns
    -------
    dict with all metrics
    """
    results = {
        'Model'     : model_name,
        'Accuracy'  : accuracy_score(y_true, y_pred),
        'Recall'    : recall_score(y_true, y_pred, zero_division=0),
        'Precision' : precision_score(y_true, y_pred, zero_division=0),
        'F1'        : f1_score(y_true, y_pred, zero_division=0),
        'F2'        : fbeta_score(y_true, y_pred, beta=2, zero_division=0),
    }
    if y_prob is not None:
        results['ROC-AUC'] = roc_auc_score(y_true, y_prob)
        results['PR-AUC']  = average_precision_score(y_true, y_prob)

    if verbose:
        print(f"\n{'='*55}")
        print(f"  {model_name}")
        print(f"{'='*55}")
        print(f"  Accuracy   : {results['Accuracy']:.4f}   <- do not use this to compare")
        print(f"  Recall     : {results['Recall']:.4f}   <- fraud caught / all fraud")
        print(f"  Precision  : {results['Precision']:.4f}   <- correct alerts / all alerts")
        print(f"  F1         : {results['F1']:.4f}")
        print(f"  F2 (b=2)   : {results['F2']:.4f}   <- emphasises recall")
        if y_prob is not None:
            print(f"  ROC-AUC    : {results['ROC-AUC']:.4f}")
            print(f"  PR-AUC     : {results['PR-AUC']:.4f}   <- key metric")
        print()
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Fraud'])
        fig, ax = plt.subplots(figsize=(4, 3))
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{model_name}', fontweight='bold', fontsize=10)
        plt.tight_layout()
        plt.show()

    return results

# Collect all results for final comparison
all_results = []


---
## 7. Baseline — Logistic Regression (No Intervention)

We start with the simplest possible model and default settings.
This baseline exposes the accuracy paradox on real data.

**Expected behaviour:**  
- Accuracy ~99% (the paradox)  
- Recall on fraud: poor — the model learns to mostly predict "legitimate"
because that minimises cross-entropy loss on the dominant class


In [ ]:
# Baseline: no class balancing, default 0.5 threshold
lr_base = LogisticRegression(max_iter=1000, random_state=42)
lr_base.fit(X_train, y_train)

y_pred_base = lr_base.predict(X_test)
y_prob_base = lr_base.predict_proba(X_test)[:, 1]

results_base = evaluate_model(
    y_test, y_pred_base, y_prob_base,
    model_name="Baseline LR (no intervention)"
)
all_results.append(results_base)


### Observation

Notice accuracy is ~99% — yet recall on fraud is low.  
The model predicts "legitimate" for almost everything and still appears excellent by accuracy alone.  
PR-AUC and ROC-AUC diverge sharply here — watch this gap narrow as we apply interventions.


---
## 8. Intervention 1 — Class Weights

**Idea:** penalise misclassification of the minority class more heavily during training.  
The loss function becomes asymmetric: getting a fraud wrong costs more.

**Formula (balanced weighting):**
```
weight_class_k = n_samples / (n_classes x n_samples_in_class_k)
```

In sklearn: `class_weight='balanced'`  
In PyTorch: `pos_weight` parameter in `BCEWithLogitsLoss`  
In XGBoost: `scale_pos_weight = n_negative / n_positive`

**Why try this first?**
- Zero data modification — no synthetic samples, no information loss
- One parameter change
- Works natively with virtually all sklearn classifiers and deep learning frameworks
- Computationally free

> Apply class weights as your default starting point before any resampling.


In [ ]:
# Compute and inspect the balanced weights
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print("Computed class weights:")
for cls, w in class_weight_dict.items():
    label = 'Fraud' if cls == 1 else 'Legitimate'
    print(f"  Class {cls} ({label}): weight = {w:.2f}")
print()
print(f"Interpretation: each fraud sample counts {weights[1]/weights[0]:.0f}x more in the loss.")


In [ ]:
# Logistic Regression with balanced class weights
lr_weighted = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_weighted.fit(X_train, y_train)

y_pred_weighted = lr_weighted.predict(X_test)
y_prob_weighted  = lr_weighted.predict_proba(X_test)[:, 1]

results_weighted = evaluate_model(
    y_test, y_pred_weighted, y_prob_weighted,
    model_name="LR + Class Weights (balanced)"
)
all_results.append(results_weighted)


---
## 9. Intervention 2 — Threshold Tuning

Every probabilistic classifier outputs a score in [0, 1].  
The **default decision threshold is 0.5**: if P(fraud) > 0.5 → predict fraud.

**The problem:** this threshold assumes equal class priors.  
With 0.17% fraud, the natural threshold is much lower.

**How to find the optimal threshold:**
1. Compute the Precision-Recall curve across all possible thresholds
2. For each threshold, compute the metric you care about (F1, F2, or a business cost ratio)
3. Pick the threshold that maximises it

**Important:** this is a free operation on an already-trained model — no retraining needed.

### Choosing the right metric to optimise
- **F1** — balanced precision/recall trade-off
- **F2** — emphasises recall (β=2 means recall matters twice as much as precision)
- **Business cost ratio** — if each missed fraud costs €500 and each false alert costs €5,
  set threshold to minimise expected cost


In [ ]:
# Compute PR curve and F-scores across all thresholds for the baseline model
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_base)

# F1 and F2 at each threshold (drop last point which has no corresponding threshold)
f1_scores = np.where(
    (precisions[:-1] + recalls[:-1]) > 0,
    2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1]),
    0
)
f2_scores = np.where(
    (4 * precisions[:-1] + recalls[:-1]) > 0,
    5 * (precisions[:-1] * recalls[:-1]) / (4 * precisions[:-1] + recalls[:-1]),
    0
)

best_idx_f1 = np.argmax(f1_scores)
best_idx_f2 = np.argmax(f2_scores)
best_threshold_f1 = thresholds[best_idx_f1]
best_threshold_f2 = thresholds[best_idx_f2]

print(f"Best threshold (max F1): {best_threshold_f1:.4f}  F1 = {f1_scores[best_idx_f1]:.4f}")
print(f"Best threshold (max F2): {best_threshold_f2:.4f}  F2 = {f2_scores[best_idx_f2]:.4f}")


In [ ]:
# Visualise threshold selection
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: PR curve coloured by threshold
sc = axes[0].scatter(recalls[:-1], precisions[:-1], c=thresholds,
                     cmap='viridis', s=5, alpha=0.7)
axes[0].scatter(recalls[best_idx_f1], precisions[best_idx_f1],
                color='red', s=120, zorder=5,
                label=f'Best F1 (t={best_threshold_f1:.3f})')
axes[0].scatter(recalls[best_idx_f2], precisions[best_idx_f2],
                color='orange', s=120, marker='D', zorder=5,
                label=f'Best F2 (t={best_threshold_f2:.3f})')
plt.colorbar(sc, ax=axes[0], label='Threshold')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve — Baseline LR', fontweight='bold')
axes[0].legend(fontsize=9)

# Right: Score vs threshold
axes[1].plot(thresholds, f1_scores, label='F1', color='#3498db', lw=2)
axes[1].plot(thresholds, f2_scores, label='F2 (beta=2)', color='#e74c3c', lw=2)
axes[1].axvline(best_threshold_f1, color='#3498db', linestyle='--', alpha=0.6)
axes[1].axvline(best_threshold_f2, color='#e74c3c',  linestyle='--', alpha=0.6)
axes[1].axvline(0.5, color='gray', linestyle=':', alpha=0.7, label='Default threshold (0.5)')
axes[1].set_xlabel('Classification Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('F1 / F2 vs Threshold', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Apply the best F2 threshold — recall-focused, appropriate for fraud detection
y_pred_tuned = (y_prob_base >= best_threshold_f2).astype(int)

results_tuned = evaluate_model(
    y_test, y_pred_tuned, y_prob_base,
    model_name=f"Baseline LR + Threshold Tuning (t={best_threshold_f2:.3f})"
)
all_results.append(results_tuned)


### Observation

Threshold tuning is a **free lunch** — same trained model, different decision boundary.  
By lowering the threshold, we increase recall (catch more fraud) at the cost of more false positives.  
F2 explicitly encodes the domain priority: recall matters twice as much as precision.

> In a real fraud system, you would set the threshold based on a business cost analysis:  
> *How much does a missed fraud cost vs. the cost of investigating a false alarm?*


---
## 10. Intervention 3 — Resampling Strategies

Resampling changes the training data distribution so the model sees a more balanced view of both classes.

### The leakage trap — and how to avoid it

A very common mistake:
```python
# WRONG: resampling before CV — leaks validation data into training
X_smote, y_smote = SMOTE().fit_resample(X_train, y_train)
cross_val_score(clf, X_smote, y_smote, cv=5)  # <- val fold data was used in SMOTE
```

**Correct approach:** use `imblearn.Pipeline`, which applies resampling **inside each fold** automatically.

```python
# CORRECT: resampling inside each fold — no leakage
pipe = ImbPipeline([('smote', SMOTE()), ('clf', LogisticRegression())])
cross_val_score(pipe, X_train, y_train, cv=StratifiedKFold(5))
```

### Resampling methods compared

| Method | Type | How it works |
|--------|------|-------------|
| `RandomOverSampler` | Oversampling | Duplicates minority samples randomly |
| `RandomUnderSampler` | Undersampling | Randomly drops majority samples |
| `SMOTE` | Oversampling | Generates synthetic samples by interpolating between minority neighbours |
| `ADASYN` | Oversampling | Like SMOTE but generates more samples where the class boundary is harder |
| `SMOTETomek` | Combined | SMOTE oversampling + Tomek link cleaning at the decision boundary |

> **Time series warning:** SMOTE ignores temporal ordering.
> On sequential data (e.g., time-series sensor readings), synthetic samples may violate causality.
> Use with caution and always validate on a held-out time window, not random folds.


In [ ]:
# Leakage-proof CV using imblearn.Pipeline + StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'recall'    : 'recall',
    'precision' : 'precision',
    'f1'        : 'f1',
    'pr_auc'    : 'average_precision',
    'roc_auc'   : 'roc_auc',
}

def cv_evaluate(sampler, sampler_name, clf=None):
    """Run stratified 5-fold CV with the given sampler inside an imblearn Pipeline."""
    if clf is None:
        clf = LogisticRegression(max_iter=1000, random_state=42)

    steps = []
    if sampler is not None:
        steps.append(('sampler', sampler))
    steps.append(('scaler', StandardScaler()))
    steps.append(('clf', clf))

    pipe = ImbPipeline(steps)
    cv_res = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)

    return {
        'Method'    : sampler_name,
        'Recall'    : cv_res['test_recall'].mean(),
        'Precision' : cv_res['test_precision'].mean(),
        'F1'        : cv_res['test_f1'].mean(),
        'PR-AUC'    : cv_res['test_pr_auc'].mean(),
        'ROC-AUC'   : cv_res['test_roc_auc'].mean(),
    }

cv_results_list = []
print("Running stratified 5-fold CV for each resampling strategy...")
print("(This may take a few minutes on the full dataset)")


In [ ]:
# Run all resampling strategies
configs = [
    (None,                                    "No Resampling (baseline)"),
    (RandomOverSampler(random_state=42),      "Random Oversampling"),
    (RandomUnderSampler(random_state=42),     "Random Undersampling"),
    (SMOTE(random_state=42),                  "SMOTE"),
    (ADASYN(random_state=42),                 "ADASYN"),
    (SMOTETomek(random_state=42),             "SMOTETomek (combined)"),
]

for i, (sampler, name) in enumerate(configs, 1):
    print(f"{i}/{len(configs)}  {name}...")
    r = cv_evaluate(sampler, name)
    cv_results_list.append(r)
    print(f"       PR-AUC={r['PR-AUC']:.3f}  Recall={r['Recall']:.3f}  F1={r['F1']:.3f}")

print("\nAll CV runs complete.")


In [ ]:
# Display CV results table
cv_df = pd.DataFrame(cv_results_list).set_index('Method').round(3)
print("=== 5-Fold Stratified CV Results — Logistic Regression ===")
print(cv_df.to_string())


In [ ]:
# Visualise resampling comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

metrics_bar = ['Recall', 'Precision', 'F1', 'PR-AUC']
colors_bar  = ['#e74c3c', '#3498db', '#2ecc71', '#8e44ad']
x = np.arange(len(cv_df))
width = 0.2

for i, (metric, color) in enumerate(zip(metrics_bar, colors_bar)):
    axes[0].bar(x + i * width, cv_df[metric], width, label=metric, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(cv_df.index, rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('Score (5-fold CV mean)')
axes[0].set_title('Resampling Methods — Metric Comparison', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1)

# PR-AUC alone for clarity
axes[1].barh(cv_df.index, cv_df['PR-AUC'], color='#8e44ad', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('PR-AUC (5-fold CV mean)')
axes[1].set_title('PR-AUC by Resampling Method\n(the honest metric for imbalanced data)',
                  fontweight='bold')
axes[1].axvline(cv_df.loc['No Resampling (baseline)', 'PR-AUC'],
                color='red', linestyle='--', alpha=0.6, label='Baseline')
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 11. Intervention 4 — Stronger Classifiers with Class Weights

Logistic Regression is linear. Real fraud patterns are non-linear.  
We now apply the best practices to stronger tree-based models.

Both **Random Forest** and **XGBoost** support class weighting natively:
- Random Forest: `class_weight='balanced'`
- XGBoost: `scale_pos_weight = n_negative / n_positive`


In [ ]:
# Random Forest — balanced class weights
rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]

results_rf = evaluate_model(
    y_test, y_pred_rf, y_prob_rf,
    model_name="Random Forest (balanced weights)"
)
all_results.append(results_rf)


In [ ]:
# XGBoost — scale_pos_weight for class imbalance
# scale_pos_weight = count(negative) / count(positive)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"XGBoost scale_pos_weight = {scale_pos_weight:.1f}")
print(f"Each fraud sample weighted {scale_pos_weight:.0f}x more in the loss.\n")

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=100,
    eval_metric='aucpr',   # optimise PR-AUC during training — right metric from the start
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb  = xgb_model.predict_proba(X_test)[:, 1]

results_xgb = evaluate_model(
    y_test, y_pred_xgb, y_prob_xgb,
    model_name="XGBoost (scale_pos_weight)"
)
all_results.append(results_xgb)


---
## 12. PR Curve & ROC Curve — All Models Overlaid

This is the definitive visual comparison.

### Reading the PR curve
- Higher area = better (more precision at every recall level)
- A random classifier on 0.17% fraud sits at PR ≈ 0.0017 (dashed line)
- PR curves spread models apart far more than ROC curves do

### Reading the ROC curve
- Closer to top-left = better
- A random classifier sits on the diagonal (AUC = 0.50)

**Notice how the ROC curves cluster together while the PR curves spread out.**  
This is not a coincidence — it is a structural property of highly imbalanced data.


In [ ]:
models_for_curves = [
    ("Baseline LR",          y_prob_base),
    ("LR + Class Weights",   y_prob_weighted),
    ("Random Forest",        y_prob_rf),
    ("XGBoost",              y_prob_xgb),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
curve_colors = ['#7f8c8d', '#e67e22', '#27ae60', '#8e44ad']
linestyles   = ['-', '--', '-.', ':']

for (name, probs), color, ls in zip(models_for_curves, curve_colors, linestyles):
    # PR curve
    p, r, _ = precision_recall_curve(y_test, probs)
    pr_auc   = average_precision_score(y_test, probs)
    axes[0].plot(r, p, color=color, lw=2, ls=ls, label=f"{name} (PR-AUC={pr_auc:.3f})")

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = roc_auc_score(y_test, probs)
    axes[1].plot(fpr, tpr, color=color, lw=2, ls=ls, label=f"{name} (ROC-AUC={roc_auc:.3f})")

# Baselines
axes[0].axhline(y_test.mean(), color='red', ls=':', alpha=0.5,
                label=f'Random classifier (PR={y_test.mean():.4f})')
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title('Precision-Recall Curves\n(key metric for imbalanced data)', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])

axes[1].plot([0, 1], [0, 1], 'r:', alpha=0.5, label='Random classifier (AUC=0.50)')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[1].set_title('ROC Curves', fontweight='bold')
axes[1].legend(fontsize=9, loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.suptitle(
    'PR curves spread models apart; ROC curves cluster — PR is the honest metric here',
    fontsize=11, style='italic', y=1.01
)
plt.tight_layout()
plt.show()


---
## 13. Feature Importance

Random Forest and XGBoost both expose feature importance — a useful sanity check.  
We expect features that correlated most with `Class` in the EDA to rank highly here.


In [ ]:
# Feature importance — Random Forest
feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns)
feat_imp_top = feat_imp.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
feat_imp_top.sort_values().plot(kind='barh', ax=ax, color='#8e44ad', alpha=0.85)
ax.set_xlabel('Feature Importance (mean decrease in impurity)')
ax.set_title('Top 15 Features — Random Forest (Balanced Weights)', fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
print(feat_imp_top.head().to_string())


---
## 14. Final Results Comparison Table

All interventions side by side on the **held-out test set**.

### How to read this table

| Column | Meaning |
|--------|---------|
| `Recall` | Primary signal — what fraction of actual fraud did we catch? |
| `Precision` | How many of our fraud alerts were correct? |
| `F2` | Recall-weighted F-score — the most appropriate scalar for fraud detection |
| `PR-AUC` | Honest overall metric — not inflated by the majority class |
| `ROC-AUC` | Included for reference — tends to be optimistic here |
| `Accuracy` | Included to illustrate the paradox — do not use for model selection |


In [ ]:
results_df = pd.DataFrame(all_results).set_index('Model').round(4)

print("=== Final Results — Held-Out Test Set ===\n")
print(results_df.to_string())
print()
print("Sort by PR-AUC to rank models honestly:")
print(results_df.sort_values('PR-AUC', ascending=False)[['Recall','Precision','F2','PR-AUC','ROC-AUC']].to_string())


In [ ]:
# Final visual summary
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics_final = ['Recall', 'Precision', 'F2', 'PR-AUC']
colors_final  = ['#e74c3c', '#3498db', '#f39c12', '#8e44ad']

for ax, metric, color in zip(axes.flat, metrics_final, colors_final):
    vals = results_df[metric].sort_values(ascending=True)
    bars = ax.barh(vals.index, vals.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} — All Interventions', fontweight='bold')
    ax.set_xlim(0, min(1.15, vals.max() * 1.2))
    for bar, val in zip(bars, vals.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=8)

plt.suptitle(
    'Summary: All Interventions vs Baseline\n(Credit Card Fraud — 0.172% positive rate)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()


---
## 15. Key Takeaways

### The intervention hierarchy — cheapest to most complex

```
1. Fix your metric        -> stop using accuracy; use PR-AUC, Recall, F-beta
2. Tune the threshold     -> free; works on any already-trained model
3. Add class weights      -> one parameter change; should be your default
4. Resample (SMOTE etc.)  -> helps at the margin; use imblearn.Pipeline + StratifiedKFold
5. Use a stronger model   -> RF / XGBoost with class weighting often dominate
6. Collect more data      -> if the above isn't enough
7. Reframe the problem    -> anomaly detection, one-class SVM, isolation forest
```

### Rules that must not be broken

| Rule | Consequence if violated |
|------|------------------------|
| Never apply SMOTE before train/test split | Data leakage — all metrics are lies |
| Always use `imblearn.Pipeline` with SMOTE in CV | Otherwise validation data leaks into resampling |
| Always use `StratifiedKFold` | Random folds may have zero minority samples |
| Report PR-AUC alongside ROC-AUC | ROC-AUC is optimistic — it hides bad minority class performance |
| Use F-beta (β > 1) when recall matters more | Explicitly encodes domain priority |

### The honest summary

Most imbalanced dataset problems don't need exotic solutions.  
They need you to:
1. Stop using accuracy
2. Tune the threshold before anything else
3. Apply class weights as your baseline
4. Use stratified CV correctly
5. Report PR-AUC alongside ROC-AUC

The sophistication is in **understanding what's happening**, not in stacking preprocessing techniques.

---

### References & further reading

- [imbalanced-learn documentation](https://imbalanced-learn.org/stable/)
- [Kaggle dataset — mlg-ulb/creditcardfraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
- Chawla et al. (2002) — *SMOTE: Synthetic Minority Over-sampling Technique*
- He et al. (2008) — *ADASYN: Adaptive Synthetic Sampling Approach for Imbalanced Learning*
- Davis & Goadrich (2006) — *The Relationship Between Precision-Recall and ROC Curves*

---

*Author: [Your Name] | [LinkedIn] | [GitHub repo link]*
